#Powered by [@CoinNoin](https://www.youtube.com/@CoinNoin)
[![Subscribe](https://img.shields.io/badge/YouTube-Subscribe%20@CoinNoin-red?style=for-the-badge&logo=youtube)](https://www.youtube.com/@CoinNoin)

In [ ]:
# @title ⚙️ 1. Install Demucs & Setup Environment
# @markdown Run this cell first to install Demucs, FFmpeg bindings, and check GPU availability.

import os
import gc
import torch
from IPython.display import clear_output

print("📦 Installing Demucs and audio processing libraries...")
!pip install -q -U demucs ffmpeg-python soundfile

# Clear GPU cache and RAM
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gpu_name = torch.cuda.get_device_name(0)
    device_status = f"✅ GPU Active: {gpu_name}"
else:
    device_status = "⚠️ No GPU detected. Running on CPU (processing will be slower)."

clear_output()
print("=" * 60)
print("✅ Demucs installation and environment setup complete!")
print(device_status)
print("=" * 60)

In [ ]:
# @title 🎛️ 2. Stem Separation & Audio Extraction (Interactive UI)
# @markdown Configure your separation settings below, then click **Run** to upload your media files and process them.

# @markdown ---
# @markdown ### 🧠 Model & Separation Mode
model_name = "htdemucs" # @param ["htdemucs", "htdemucs_ft", "htdemucs_6s", "mdx_extra"] {presence: "required"}
separation_mode = "All 4 Stems (Vocals, Drums, Bass, Other)" # @param ["Two-stems: Vocals + Instrumental", "Two-stems: Drums + No Drums", "Two-stems: Bass + No Bass", "Two-stems: Other + No Other", "All 4 Stems (Vocals, Drums, Bass, Other)", "All 6 Stems (htdemucs_6s only: Vocals, Drums, Bass, Other, Guitar, Piano)"]

# @markdown ---
# @markdown ### 🔊 Audio Output Settings
output_format = "mp3" # @param ["mp3", "wav", "flac"]
mp3_bitrate = 320 # @param [128, 192, 256, 320]
boost_background_db = 6 # @param {type:"slider", min:0, max:20, step:1}
# @markdown *Guide: Set boost to 0 for original levels. Higher values (e.g., 6–15 dB) amplify quiet backing tracks with a soft limiter.*

# @markdown ---
# @markdown ### ⚡ Quality & Performance Tuning
segment_size = 7 # @param {type:"slider", min:4, max:15, step:1}
# @markdown *Guide: 7 is the default and prevents Transformer RAM overflow on Colab T4 GPUs.*
shifts = 1 # @param {type:"slider", min:1, max:5, step:1}
# @markdown *Guide: Number of random shifts for equivariant stabilization. 1 is fastest; 2–4 gives slightly higher quality at the cost of processing time.*
auto_download_zip = True # @param {type:"boolean"}

# ============================================================
# RUNTIME PROCESSING
# ============================================================
import os
import shutil
import subprocess
import zipfile
import gc
import torch
from google.colab import files
from IPython.display import Audio, display, clear_output

# Base paths
WORKDIR = "/content/demucs_workspace"
INPUT_DIR = os.path.join(WORKDIR, "uploads")
TEMP_DIR = os.path.join(WORKDIR, "temp")
OUTPUT_DIR = os.path.join(WORKDIR, "separated_stems")
FINAL_EXPORT_DIR = os.path.join(WORKDIR, "final_output")

# Clean existing workspace to keep RAM and disk fresh
for folder in [TEMP_DIR, OUTPUT_DIR, FINAL_EXPORT_DIR]:
    if os.path.exists(folder):
        shutil.rmtree(folder)
    os.makedirs(folder, exist_ok=True)
os.makedirs(INPUT_DIR, exist_ok=True)

print("=" * 60)
print("📤 UPLOAD YOUR MEDIA (Audio or Video)")
print("Supports: .mp3, .wav, .m4a, .flac, .aac, .ogg, .opus, .mp4, .mkv, .mov, .avi, .webm, .flv")
print("=" * 60)

uploaded = files.upload()

if not uploaded:
    print("\n❌ No files were uploaded. Please run this cell again and select at least one file.")
else:
    saved_files = []
    for filename in uploaded.keys():
        dest_path = os.path.join(INPUT_DIR, filename)
        # Move uploaded file to INPUT_DIR
        if os.path.exists(filename) and filename != dest_path:
            shutil.move(filename, dest_path)
        saved_files.append(dest_path)

    clear_output()
    print(f"✅ {len(saved_files)} file(s) received. Starting separation...\n")

    # Map separation mode to CLI arguments
    two_stem_flags = []
    if separation_mode == "Two-stems: Vocals + Instrumental":
        two_stem_flags = ["--two-stems", "vocals"]
    elif separation_mode == "Two-stems: Drums + No Drums":
        two_stem_flags = ["--two-stems", "drums"]
    elif separation_mode == "Two-stems: Bass + No Bass":
        two_stem_flags = ["--two-stems", "bass"]
    elif separation_mode == "Two-stems: Other + No Other":
        two_stem_flags = ["--two-stems", "other"]

    # Select effective model
    selected_model = model_name
    if "6 Stems" in separation_mode and model_name != "htdemucs_6s":
        print("ℹ️ Switching model to 'htdemucs_6s' to extract 6 stems.")
        selected_model = "htdemucs_6s"

    exported_files_for_preview = []

    for idx, input_path in enumerate(saved_files, start=1):
        filename = os.path.basename(input_path)
        base_name = os.path.splitext(filename)[0]

        print("=" * 70)
        print(f"[{idx}/{len(saved_files)}] PROCESSING: {filename}")
        print("=" * 70)

        # 1. Convert video/audio to standard 44.1kHz 16-bit WAV for Demucs
        temp_wav = os.path.join(TEMP_DIR, f"{base_name}_temp.wav")
        print("\n[1/3] 🔄 Extracting and normalizing audio track...")

        conv_cmd = [
            "ffmpeg", "-y", "-i", input_path,
            "-vn", "-ac", "2", "-ar", "44100",
            "-c:a", "pcm_s16le", temp_wav,
            "-loglevel", "error"
        ]
        res = subprocess.run(conv_cmd, capture_output=True, text=True)
        if res.returncode != 0 or not os.path.exists(temp_wav):
            print(f"❌ Failed to extract audio track from {filename}:\n{res.stderr}")
            continue

        # 2. Run Demucs separation
        print(f"\n[2/3] 🧠 Separating stems with model '{selected_model}'...")
        demucs_cmd = [
            "python", "-m", "demucs",
            "-n", selected_model,
            "--segment", str(segment_size),
            "--shifts", str(shifts),
            "-o", OUTPUT_DIR,
            temp_wav
        ] + two_stem_flags

        if output_format == "mp3":
            demucs_cmd.extend(["--mp3", "--mp3-bitrate", str(mp3_bitrate)])
        elif output_format == "flac":
            demucs_cmd.append("--flac")

        res = subprocess.run(demucs_cmd, capture_output=True, text=True)
        if res.returncode != 0:
            print(f"❌ Demucs separation failed:\n{res.stderr}")
            continue

        # Locate Demucs output directory
        target_stem_dir = os.path.join(OUTPUT_DIR, selected_model, f"{base_name}_temp")
        if not os.path.exists(target_stem_dir):
            print(f"❌ Could not find output directory: {target_stem_dir}")
            continue

        # 3. Post-processing & volume adjustments
        print("\n[3/3] 🎚️ Finalizing stems & applying sound levels...")
        item_output_dir = os.path.join(FINAL_EXPORT_DIR, base_name)
        os.makedirs(item_output_dir, exist_ok=True)

        for stem_file in os.listdir(target_stem_dir):
            stem_src = os.path.join(target_stem_dir, stem_file)
            stem_name, stem_ext = os.path.splitext(stem_file)
            final_filename = f"{base_name}_{stem_name}{stem_ext}"
            final_dst = os.path.join(item_output_dir, final_filename)

            # Apply volume boost to background/instrumental tracks if requested
            is_backing = stem_name in ["no_vocals", "no_drums", "no_bass", "no_other", "other"]
            if boost_background_db > 0 and is_backing:
                boost_cmd = [
                    "ffmpeg", "-y", "-i", stem_src,
                    "-filter:a", f"volume={boost_background_db}dB,alimiter=limit=0.95",
                    final_dst, "-loglevel", "error"
                ]
                subprocess.run(boost_cmd)
            else:
                shutil.copy2(stem_src, final_dst)

            exported_files_for_preview.append(final_dst)
            print(f"  ✨ Exported: {final_filename}")

        # Cleanup intermediate temporary files for this item
        if os.path.exists(temp_wav):
            os.remove(temp_wav)
        if os.path.exists(target_stem_dir):
            shutil.rmtree(target_stem_dir)

        # Clear PyTorch cache
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # ============================================================
    # 4. PACKAGE AND DOWNLOAD
    # ============================================================
    print("\n" + "=" * 70)
    print("🎉 ALL PROCESSING COMPLETED!")
    print("=" * 70)

    # Create ZIP file containing all separated stems
    zip_path = "/content/Separated_Stems_Demucs-CoinNoin.zip"
    if os.path.exists(zip_path):
        os.remove(zip_path)

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for root, _, filenames in os.walk(FINAL_EXPORT_DIR):
            for f in filenames:
                full_path = os.path.join(root, f)
                arcname = os.path.relpath(full_path, FINAL_EXPORT_DIR)
                z.write(full_path, arcname)

    print(f"\n📦 All separated tracks have been packaged into: {zip_path}")

    # Display audio players for immediate listening
    print("\n🎧 Stem Preview Players:")
    for preview_file in exported_files_for_preview[:6]:
        print(f"\n▶️ {os.path.basename(preview_file)}")
        display(Audio(preview_file))

    # Automatic download trigger
    if auto_download_zip:
        print("\n⬇️ Starting automatic download of Separated_Stems.zip...")
        files.download(zip_path)